# Day 007：Multi-Head Attention 的拆分与组合

本 Notebook 与 [Day 007 互动档案](../day-007.md) 配套。它从完整输入 `h` 开始，依次运行三套投影、拆 Head、每个 Head 的 Attention、按 token 拼接和 `o_proj`。

当前只实现普通 Multi-Head Attention：Q/K/V 的 Head 数相同。MiniMind-3 的 GQA（8 个 Q Head、4 个 K/V Head）留到下一阶段。

## 1. 完整输入先投影，再拆 Head

配置：`batch=1`、`sequence=3`、`hidden_size=8`、`num_heads=2`、`head_dim=4`。第一层中的 `h` 可以来自 Embedding；后续层中的 `h` 来自上一层。

In [ ]:
import torch
from torch import nn

torch.manual_seed(7)
torch.set_printoptions(precision=4, sci_mode=False)

batch = 1
sequence = 3
hidden_size = 8
num_heads = 2
head_dim = 4

assert hidden_size == num_heads * head_dim

h = torch.arange(
    batch * sequence * hidden_size,
    dtype=torch.float32
).reshape(batch, sequence, hidden_size)

q_proj = nn.Linear(hidden_size, hidden_size, bias=False)
k_proj = nn.Linear(hidden_size, hidden_size, bias=False)
v_proj = nn.Linear(hidden_size, hidden_size, bias=False)

q_projected = q_proj(h)
k_projected = k_proj(h)
v_projected = v_proj(h)

def split_heads(projected):
    return projected.reshape(
        batch, sequence, num_heads, head_dim
    ).transpose(1, 2)

q_heads = split_heads(q_projected)
k_heads = split_heads(k_projected)
v_heads = split_heads(v_projected)

print("h：", h.shape)
print("q/k/v projected：", q_projected.shape)
print("q/k/v heads：", q_heads.shape)
print("投影是否改变数值：", not torch.allclose(h, q_projected))

q_restored = q_heads.transpose(1, 2).reshape(batch, sequence, hidden_size)
print("拆分后能否无损拼回：", torch.allclose(q_restored, q_projected))
print("Q/K 是否因投影权重不同而不同：", not torch.allclose(q_projected, k_projected))

shape 链：

```text
h / projected   [B, S, hidden_size] = [1, 3, 8]
reshape         [B, S, H, D]         = [1, 3, 2, 4]
transpose       [B, H, S, D]         = [1, 2, 3, 4]
```

投影由 Linear 改变数值；reshape/transpose 只重新组织数值。

## 2. 两个 Head 批量独立计算 Attention

`q_heads @ k_heads.transpose(-2, -1)` 会保留 batch 和 head 维，分别在每个 Head 内计算 `sequence × sequence` 分数矩阵。

In [ ]:
scores = (q_heads @ k_heads.transpose(-2, -1)) / (head_dim ** 0.5)

causal_mask = torch.triu(
    torch.ones(sequence, sequence, dtype=torch.bool),
    diagonal=1
)
masked_scores = scores.masked_fill(causal_mask, float("-inf"))
weights = torch.softmax(masked_scores, dim=-1)

print("scores shape：", scores.shape)
print("weights shape：", weights.shape)
print("每个 Head 的每行比例之和：")
print(weights.sum(dim=-1))
print("未来位置是否全部为 0：", torch.all(weights.masked_select(causal_mask) == 0).item())
print("Head 0 读取比例：")
print(weights[0, 0])
print("Head 1 读取比例：")
print(weights[0, 1])

这里不需要 Python `for` 循环：shape `[B,H,S,D]` 中的 `H` 维让两个 Head 被批量计算，但不同 Head 的 Q/K/V 仍不会互相混合。

## 3. 各 Head 读取 V，再按 token 拼接

每个 Head 的 `[S,S]` 权重矩阵读取同编号的 `[S,D]` V，得到 `[S,D]`。然后先交换回 `[B,S,H,D]`，再把 `H×D` 合成 `hidden_size`。

In [ ]:
head_results = weights @ v_heads
combined = head_results.transpose(1, 2).reshape(
    batch, sequence, hidden_size
)

print("每个 Head 的结果：", head_results.shape)
print("按 token 拼接后：", combined.shape)
print("第一个 token 的 Head 0 结果：", head_results[0, 0, 0])
print("第一个 token 的 Head 1 结果：", head_results[0, 1, 0])
print("第一个 token 拼接结果：", combined[0, 0])

反向组织过程：

```text
head_results   [B, H, S, D]
transpose      [B, S, H, D]
reshape        [B, S, H*D] = [B, S, hidden_size]
```

## 4. `o_proj` 学习混合各 Head 的信息

拼接后 shape 已经恢复为 `hidden_size`，但各 Head 仍只是分段排列。`o_proj` 让每个输出维度使用全部 Head 的结果。

In [ ]:
o_proj = nn.Linear(hidden_size, hidden_size, bias=False)
attention_output = o_proj(combined)

print("combined shape：", combined.shape)
print("attention_output shape：", attention_output.shape)
print("o_proj 前后数值是否相同：", torch.allclose(combined, attention_output))
print("第一个 token 的最终 Attention 输出：")
print(attention_output[0, 0])

## 5. 完整 shape 总结

```text
h                            [B,S,hidden]
q/k/v_proj                   [B,S,hidden]
split + transpose            [B,H,S,D]
Q @ K.T                      [B,H,S,S]
Softmax 后 weights           [B,H,S,S]
weights @ V                  [B,H,S,D]
transpose + reshape          [B,S,H*D] = [B,S,hidden]
o_proj                       [B,S,hidden]
```

下一阶段再学习 MiniMind-3 为什么使用 8 个 Q Head、4 个 K/V Head，以及数量不同的 Head 怎样配对。